# Resume Coach - Prompt Engineering & Parser Demo

This notebook demonstrates the Resume Parser, Job Parser, and LLM integration for generating coaching reports.

## Overview
- Parse resumes from PDF/text
- Parse job descriptions
- Generate AI coaching reports
- Test prompt engineering strategies

## 1. Import Required Libraries

In [ ]:
import sys
import os
from pathlib import Path

# Add parent directory to path to import backend modules
sys.path.insert(0, str(Path().resolve().parent))

from backend.services.resume_parser import ResumeParser
from backend.services.job_parser import JobParser
from backend.services.llm_service import LLMService
from backend.models.schema import Resume, JobDescription
from backend.prompts.templates import PromptTemplates

from dotenv import load_dotenv
import json

# Load environment variables
load_dotenv(Path().resolve().parent / '.env')

print("✅ All modules imported successfully!")
print(f"OpenAI API Key configured: {bool(os.getenv('OPENAI_API_KEY'))}")

## 2. Sample Data - Resume Text

In [ ]:
sample_resume = """
SARAH JOHNSON
sarah.johnson@email.com | (415) 555-0123 | linkedin.com/in/sarahjohnson
San Francisco, CA

PROFESSIONAL SUMMARY
Senior Software Engineer with 7+ years of experience building scalable web applications.
Expert in Python, JavaScript, and cloud technologies. Passionate about clean code and mentoring.

TECHNICAL SKILLS
Languages: Python, JavaScript, TypeScript, Java, SQL
Frameworks: Django, Flask, React, Node.js, FastAPI
Cloud & DevOps: AWS (EC2, S3, Lambda), Docker, Kubernetes, CI/CD
Databases: PostgreSQL, MongoDB, Redis
Tools: Git, Jenkins, Terraform, VS Code

PROFESSIONAL EXPERIENCE

Senior Software Engineer | TechCorp Inc. | San Francisco, CA
March 2021 - Present
• Led development of microservices architecture serving 2M+ users
• Reduced API response time by 45% through optimization and caching
• Mentored team of 4 junior developers
• Implemented automated testing, increasing code coverage from 60% to 95%
• Collaborated with product managers and designers on feature planning

Software Engineer | StartupXYZ | Mountain View, CA
July 2018 - February 2021
• Built RESTful APIs using Python/Django and PostgreSQL
• Developed React-based admin dashboard
• Implemented authentication and authorization system
• Participated in code reviews and agile sprint planning

Junior Developer | WebSolutions | Palo Alto, CA
June 2016 - June 2018
• Maintained legacy PHP applications
• Migrated applications to Python/Flask
• Fixed bugs and implemented new features
• Wrote technical documentation

EDUCATION

Bachelor of Science in Computer Science
Stanford University | Graduated May 2016 | GPA: 3.8/4.0

CERTIFICATIONS
• AWS Certified Solutions Architect - Associate
• Professional Scrum Master (PSM I)

PROJECTS
• Open-source contributor to Django REST Framework (500+ stars)
• Personal project: AI-powered code review tool (Python + GPT-4)
"""

print("Sample resume loaded ✅")
print(f"Resume length: {len(sample_resume)} characters")

## 3. Sample Data - Job Description

In [ ]:
sample_job_description = """
Senior Software Engineer - Backend

TechCorp Inc.
San Francisco, CA (Hybrid)
$150,000 - $200,000

About the Role:
We're looking for a Senior Software Engineer to join our Backend Infrastructure team. You'll design and build scalable microservices that power our platform serving millions of users.

Responsibilities:
- Design and implement high-performance RESTful APIs
- Build microservices using Python and Go
- Optimize database queries and caching strategies
- Mentor junior engineers and conduct code reviews
- Collaborate with DevOps on CI/CD pipelines
- Participate in on-call rotation

Required Qualifications:
- 5+ years of software engineering experience
- Strong proficiency in Python or Go
- Experience with microservices architecture
- Expertise in SQL databases (PostgreSQL, MySQL)
- Experience with cloud platforms (AWS, GCP, or Azure)
- Strong understanding of RESTful API design
- Bachelor's degree in Computer Science or related field

Preferred Qualifications:
- Experience with Kubernetes and Docker
- Knowledge of message queues (RabbitMQ, Kafka)
- Familiarity with Redis or Memcached
- Experience with monitoring tools (Datadog, New Relic)
- Contributions to open-source projects
- Master's degree in Computer Science

Tech Stack:
Python, Go, PostgreSQL, Redis, Docker, Kubernetes, AWS, Terraform

Benefits:
- Competitive salary and equity
- Health, dental, and vision insurance
- 401(k) matching
- Unlimited PTO
- Remote work flexibility
- Professional development budget
"""

print("Sample job description loaded ✅")
print(f"Length: {len(sample_job_description)} characters")

## 4. Test Resume Parser

In [ ]:
# Initialize Resume Parser
resume_parser = ResumeParser()

# Parse the sample resume text
parsed_resume = resume_parser.parse_text(sample_resume)

print("✅ Resume Parsed Successfully!\n")
print("=" * 60)
print(f"Name: {parsed_resume.contact_info.name}")
print(f"Email: {parsed_resume.contact_info.email}")
print(f"Phone: {parsed_resume.contact_info.phone}")
print(f"LinkedIn: {parsed_resume.contact_info.linkedin}")
print(f"\nSkills ({len(parsed_resume.skills)} found):")
for skill in parsed_resume.skills[:10]:  # Show first 10
    print(f"  • {skill}")
print(f"\nExperience ({len(parsed_resume.experience)} positions):")
for exp in parsed_resume.experience:
    print(f"  • {exp.title} at {exp.company} ({exp.start_date} - {exp.end_date})")
print(f"\nEducation ({len(parsed_resume.education)} entries):")
for edu in parsed_resume.education:
    print(f"  • {edu.degree} in {edu.field_of_study} from {edu.institution} ({edu.graduation_date})")
print("=" * 60)

## 5. Test Job Description Parser

In [ ]:
# Initialize Job Parser
job_parser = JobParser()

# Parse the sample job description
parsed_job = job_parser.parse_job_description(sample_job_description)

print("✅ Job Description Parsed Successfully!\n")
print("=" * 60)
print(f"Title: {parsed_job.title}")
print(f"Company: {parsed_job.company}")
print(f"Location: {parsed_job.location}")
print(f"Salary Range: {parsed_job.salary_range}")
print(f"Job Type: {parsed_job.job_type}")
print(f"Experience Level: {parsed_job.experience_level}")
print(f"\nRequired Skills ({len(parsed_job.required_skills)} found):")
for skill in parsed_job.required_skills[:10]:  # Show first 10
    print(f"  • {skill}")
print(f"\nPreferred Skills ({len(parsed_job.preferred_skills)} found):")
for skill in parsed_job.preferred_skills[:10]:  # Show first 10
    print(f"  • {skill}")
print(f"\nKey Responsibilities ({len(parsed_job.responsibilities)} found):")
for i, resp in enumerate(parsed_job.responsibilities[:5], 1):  # Show first 5
    print(f"  {i}. {resp}")
print("=" * 60)

## 6. Generate Coaching Report (Single-Step Approach)

In [ ]:
# Initialize LLM Service with OpenAI
llm_service = LLMService(provider="openai")

print("Generating coaching report (single-step approach)...")
print("This may take 10-30 seconds...\n")

# Generate comprehensive coaching report
coaching_report = llm_service.generate_coaching_report(
    resume=parsed_resume,
    job_description=parsed_job,
    session_id="demo-session-1"
)

print("=" * 60)
print("📊 COACHING REPORT - SINGLE-STEP APPROACH")
print("=" * 60)
print(f"\n🎯 Overall Fit Score: {coaching_report.overall_fit_score}/100")
print(f"📝 Summary:\n{coaching_report.summary}\n")

print("\n❌ Gaps & Areas for Improvement:")
for i, gap in enumerate(coaching_report.gaps, 1):
    print(f"\n  {i}. {gap.skill_area} (Severity: {gap.severity})")
    print(f"     Impact: {gap.impact}")
    print(f"     Suggestion: {gap.suggestion}")

print("\n\n✅ Key Strengths:")
for i, strength in enumerate(coaching_report.strengths, 1):
    print(f"\n  {i}. {strength.skill_area} (Relevance: {strength.relevance})")
    print(f"     Evidence: {strength.evidence}")
    print(f"     Advice: {strength.how_to_highlight}")

print("\n\n💡 Actionable Recommendations:")
for i, rec in enumerate(coaching_report.recommendations, 1):
    print(f"  {i}. [{rec.priority}] {rec.action}")
    if rec.timeline:
        print(f"     Timeline: {rec.timeline}")

print("\n" + "=" * 60)

## 7. Generate Coaching Report (Multi-Step Approach)

In [ ]:
print("Generating coaching report (multi-step approach)...")
print("This approach breaks down the analysis into 5 specialized steps:")
print("  1️⃣  Fit Assessment")
print("  2️⃣  Gap Analysis")
print("  3️⃣  Strength Analysis")
print("  4️⃣  Resume Optimization")
print("  5️⃣  Interview Preparation")
print("\nThis may take 30-60 seconds...\n")

# Generate comprehensive coaching report using multi-step approach
coaching_report_multi = llm_service.generate_coaching_report_multi_step(
    resume=parsed_resume,
    job_description=parsed_job,
    session_id="demo-session-2"
)

print("=" * 60)
print("📊 COACHING REPORT - MULTI-STEP APPROACH")
print("=" * 60)
print(f"\n🎯 Overall Fit Score: {coaching_report_multi.overall_fit_score}/100")
print(f"📝 Summary:\n{coaching_report_multi.summary}\n")

print("\n❌ Gaps & Areas for Improvement:")
for i, gap in enumerate(coaching_report_multi.gaps, 1):
    print(f"\n  {i}. {gap.skill_area} (Severity: {gap.severity})")
    print(f"     Impact: {gap.impact}")
    print(f"     Suggestion: {gap.suggestion}")

print("\n\n✅ Key Strengths:")
for i, strength in enumerate(coaching_report_multi.strengths, 1):
    print(f"\n  {i}. {strength.skill_area} (Relevance: {strength.relevance})")
    print(f"     Evidence: {strength.evidence}")
    print(f"     Advice: {strength.how_to_highlight}")

print("\n\n💡 Actionable Recommendations:")
for i, rec in enumerate(coaching_report_multi.recommendations, 1):
    print(f"  {i}. [{rec.priority}] {rec.action}")
    if rec.timeline:
        print(f"     Timeline: {rec.timeline}")

print("\n" + "=" * 60)

## 8. Test Chatbot Conversation

In [ ]:
# Simulate a conversation with the career coaching chatbot
conversation_history = []

# Question 1: Ask about interview preparation
print("=" * 60)
print("💬 CHATBOT CONVERSATION TEST")
print("=" * 60)

question1 = "What are the top 3 technical questions I should prepare for this role?"
print(f"\n👤 User: {question1}\n")

response1 = llm_service.chat_response(
    message=question1,
    conversation_history=conversation_history,
    resume=parsed_resume,
    job_description=parsed_job,
    session_id="demo-chat-1"
)

print(f"🤖 Coach: {response1}\n")
conversation_history.append({"role": "user", "content": question1})
conversation_history.append({"role": "assistant", "content": response1})

# Question 2: Ask about specific skill gap
question2 = "How can I quickly improve my Kubernetes skills before the interview?"
print(f"\n👤 User: {question2}\n")

response2 = llm_service.chat_response(
    message=question2,
    conversation_history=conversation_history,
    resume=parsed_resume,
    job_description=parsed_job,
    session_id="demo-chat-1"
)

print(f"🤖 Coach: {response2}\n")
conversation_history.append({"role": "user", "content": question2})
conversation_history.append({"role": "assistant", "content": response2})

# Question 3: Ask about resume optimization
question3 = "Should I emphasize my Python or JavaScript experience more for this role?"
print(f"\n👤 User: {question3}\n")

response3 = llm_service.chat_response(
    message=question3,
    conversation_history=conversation_history,
    resume=parsed_resume,
    job_description=parsed_job,
    session_id="demo-chat-1"
)

print(f"🤖 Coach: {response3}\n")
print("=" * 60)

## 9. Comparison: Single-Step vs Multi-Step Approach

The notebook above demonstrates two different prompt engineering strategies:

### Single-Step Approach
- **Pros:** 
  - Faster execution (one API call)
  - Lower cost
  - Simpler implementation
- **Cons:**
  - May miss nuanced details
  - Limited depth in each area
  - Can be overwhelmed by complex requirements

### Multi-Step Approach
- **Pros:**
  - More detailed and comprehensive analysis
  - Better structured output
  - Each step focuses on specific aspect
  - Higher quality insights
- **Cons:**
  - Slower execution (5 API calls)
  - Higher cost
  - More complex error handling

### Recommendation
For production deployment:
- Use **Single-Step** for quick previews and demos
- Use **Multi-Step** for full coaching reports and premium features
- Consider user subscription tier when choosing approach